In [ ]:
""" Data-Construc.ipynb
This notebook provides the whole CodeMem Datasets Construction Process.
Now using filtered data :
data/
    instance_qa.json        # LLM-Generated QA for each target instance, only has 10 instance now as a attempt
    file_dependency.json    # Instance Dependency file as a Matrix, filtered by Cross-Commit-Consistency and Touched-File

Now Output Datasets :
data/
    code_mem_dataset.json

    # Only has ["Revert"] attribution now
    # Currently I put whole SWE-Gym instance into out dataset, so it might be a huge file. 

"""
from collections import Counter
import re
import json

import pandas as pd
from datasets import load_dataset

from tqdm import tqdm

from constants.config import init_env_and_logger

logger = init_env_and_logger()
# === STEP 1 : Load the SWE-Gym origin datasets and File-Dependency dict ===
SWE_DATASET_ID = "SWE-Gym/SWE-Gym"
CONFIG_NAME = None
SPLIT = "train"

load_kwargs = {"split": SPLIT}
if CONFIG_NAME is not None:
    load_kwargs["name"] = CONFIG_NAME

logger.info("Now Loading the origin SWE-gym dataset")
dataset = load_dataset(SWE_DATASET_ID, **load_kwargs)
ds = dataset.to_pandas()
print(ds)

In [16]:
# Build the instance_id to instance dict

id_map = {
    row.instance_id: row._asdict()
    for row in ds.itertuples()
}

for k,v in id_map.items():
    v["FAIL_TO_PASS"] = v["FAIL_TO_PASS"].tolist()
    v["PASS_TO_PASS"] = v["PASS_TO_PASS"].tolist()


In [17]:
# Load the File-Dependency dict and QA dict

from constants.addr import FILE_DEPENDENCY_ADDR, INSTANCE_QA_ADDR, load_json

logger.info("Loading the File-Dependency dict")
fd = load_json(FILE_DEPENDENCY_ADDR)
qa = load_json(INSTANCE_QA_ADDR)

print(json.dumps(qa, indent = 4))

2026-07-08 14:12:53,475 - INFO - Loading the File-Dependency dict


{
    "getmoto__moto-7365": {
        "questions": [
            "In the Item.update_with_attribute_updates method, what is the default update action applied when an attribute update does not explicitly specify an 'Action' field?",
            "When handling the 'ADD' action for a numeric ('N') attribute in Item.update_with_attribute_updates, what types and operations are used to compute the updated value?",
            "In the 'ADD' branch for numeric attributes in Item.update_with_attribute_updates, what default value is used when the attribute does not already exist in self.attrs?",
            "How does the Item.update_with_attribute_updates method handle the 'ADD' action when the attribute value type is a list ('L')?",
            "In the DynamoType.__add__ method, how are numeric string values converted before addition, and what form is used to construct the resulting DynamoDB NUMBER value?"
        ],
        "gold_answers": [
            "The default action is 'PUT' (Action def

In [18]:
import copy
# === step 2 : Construct dataset of "Revert" ===

# Get the repos

repos = set(ds["repo"].to_list())
repos.discard("iterative/dvc")
print(repos)

# initialize out dataset
from datetime import datetime

code_mem = {
    "metadata":{
        "name": "CodeMem",
        "version": "1.0.0",
        "generation_data": datetime.now().isoformat() 
    },
    "tasks":{
        "Revert": [],
        "User_Instructions": [] 
    }
}

logger.info("Building the Revert Tasks")
middle_num_list = [1,2,4,8,16]
Revert_Tasks_List = []

""" Revert_Tasks_List structure
[tasks]
task:{
    "repo": repo_name,
    "target": target_instance,
    "middle": [middle_instance_sequence],
    "num_middle": number of middle instance,
    # TODO
    "questions": [target questions],
    "gold_answers": [target answers]
}
"""

for repo_name, matrix in fd.items():
    # Target
    for target_id, middle_dict in matrix.items():
        # initialize task
        task = {
            "repo": repo_name,
            "target": id_map[target_id],
            "middle": [],
            "num_middle": 0,
            "questions": [],
            "gold_answers": []
        }
        if target_id in qa and len(qa[target_id]) > 0:
            task["questions"] = qa[target_id]["questions"] 
            task["gold_answers"] = qa[target_id]["gold_answers"]

        # middle
        for middle_id, flag in middle_dict.items():
            if flag == 0:
                continue
            task["middle"].append(id_map[middle_id])
            task["num_middle"] += 1
            if task["num_middle"] in middle_num_list:
                Revert_Tasks_List.append(copy.deepcopy(task))

logger.info("Revert Tasks Complete")
code_mem["tasks"]["Revert"] = Revert_Tasks_List

from constants.addr import CODE_MEM_ADDR,save_json
save_json(CODE_MEM_ADDR, code_mem)
logger.info("Save code_mem successfully!")

print(code_mem["metadata"])
# print(len(Revert_Tasks_List))
# print(Revert_Tasks_List[5]["num_middle"])









2026-07-08 14:12:56,180 - INFO - Building the Revert Tasks


{'dask/dask', 'pandas-dev/pandas', 'pydantic/pydantic', 'modin-project/modin', 'facebookresearch/hydra', 'Project-MONAI/MONAI', 'conan-io/conan', 'getmoto/moto', 'bokeh/bokeh', 'python/mypy'}


2026-07-08 14:12:57,604 - INFO - Revert Tasks Complete
2026-07-08 14:13:00,155 - INFO - Save code_mem successfully!


{'name': 'CodeMem', 'version': '1.0.0', 'generation_data': '2026-07-08T14:12:56.180110'}
